In [1]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
# Load and prepare the same dataset used in Generate_Adversarial.ipynb
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

df_sample = pd.read_csv("adv_sample_225000.csv")
print(f"Dataset shape: {df_sample.shape}")
print(df_sample["label"].value_counts())

Device: cpu
Dataset shape: (225000, 90)
label
BENIGN            75000
DoS_ATTACK        75000
NON_DoS_ATTACK    75000
Name: count, dtype: int64


In [3]:
# Encode labels and fit scaler (identical to Generate_Adversarial.ipynb)
le = LabelEncoder()
df_sample["label"] = le.fit_transform(df_sample["label"])
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))
print("Label mapping:", label_mapping)  # BENIGN=0, DoS_ATTACK=1, NON_DoS_ATTACK=2

X = df_sample.drop(columns=["label"], errors="ignore").select_dtypes(include=[np.number])
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)
y = df_sample["label"]

X_min_raw = X.min(axis=0).values
X_max_raw = X.max(axis=0).values
X_min_df = pd.DataFrame([X_min_raw], columns=X.columns)
X_max_df = pd.DataFrame([X_max_raw], columns=X.columns)

_, X_val, _, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
scaler.fit(X.loc[~X.index.isin(X_val.index)])  # fit on train split only
X_scaled = scaler.transform(X)

x_min_t = torch.tensor(scaler.transform(X_min_df).flatten(), dtype=torch.float32, device=device)
x_max_t = torch.tensor(scaler.transform(X_max_df).flatten(), dtype=torch.float32, device=device)

Label mapping: {'BENIGN': 0, 'DoS_ATTACK': 1, 'NON_DoS_ATTACK': 2}


In [4]:
# Load the pretrained DNN (same architecture as Generate_Adversarial.ipynb)
class MultiClassDNN(nn.Module):
    def __init__(self, input_size=88, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model = MultiClassDNN().to(device)
model.load_state_dict(torch.load("multiclass_dnn.pth", map_location=device))
model.eval()
print("Model loaded.")

Model loaded.


C:\Users\Aristophanies\AppData\Local\Temp\ipykernel_25568\723141856.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("multiclass_dnn.pth

In [5]:
# Isolate BENIGN samples that the model currently classifies correctly
# Feature indices (must match Generate_Adversarial.ipynb)
cont_idx = [0, 1, 2, 3, 4]
bin_idx   = [5, 6, 7]
cat_groups = [[8, 9, 10], [11, 12, 13]]

benign_mask = (y.values == 0)
X_benign = X_scaled[benign_mask]
y_benign = y.values[benign_mask]

X_benign_t = torch.tensor(X_benign, dtype=torch.float32, device=device)
y_benign_t = torch.tensor(y_benign, dtype=torch.long, device=device)

with torch.no_grad():
    preds = model(X_benign_t).argmax(dim=1).cpu().numpy()

correctly_classified = (preds == 0)
X_benign_t = X_benign_t[correctly_classified]
y_benign_t = y_benign_t[correctly_classified]

print(f"Total BENIGN samples  : {len(y_benign):,}")
print(f"Correctly classified  : {correctly_classified.sum():,}  ({correctly_classified.mean()*100:.1f}%)")
print(f"Will attack           : {len(X_benign_t):,} samples")

Total BENIGN samples  : 75,000
Correctly classified  : 75,000  (100.0%)
Will attack           : 75,000 samples


In [6]:
def targeted_pgd(model, x, y_true, target_class, epsilon,
                 x_min_t, x_max_t,
                 num_steps=100, n_restarts=10, alpha=None, momentum=0.9):
    """
    Targeted PGD: perturb x so the model predicts target_class.
    Only continuous features (cont_idx) are perturbed.
    Returns perturbed tensor.
    """
    if alpha is None:
        alpha = epsilon / 7

    model.eval()
    cont_idx_t = torch.tensor(cont_idx, device=device)
    N = x.size(0)

    best_is_fooled = torch.zeros(N, dtype=torch.bool, device=device)
    best_loss      = torch.full((N,), -float("inf"), device=device)
    best_x_adv     = x.clone()

    for restart in range(n_restarts):
        delta = torch.zeros_like(x)
        if restart > 0:
            delta[:, cont_idx_t] = torch.empty(N, len(cont_idx), device=device).uniform_(-epsilon, epsilon)

        x_adv = torch.clamp(x + delta, x_min_t, x_max_t).clone()
        grad_momentum = torch.zeros_like(x)
        still_alive = torch.ones(N, dtype=torch.bool, device=device)

        for step in range(num_steps):
            if still_alive.sum() == 0:
                break

            x_adv_alive = x_adv[still_alive].detach().requires_grad_(True)
            logits = model(x_adv_alive)

            t = torch.full((still_alive.sum(),), target_class, dtype=torch.long, device=device)
            loss = -F.cross_entropy(logits, t)   # minimize CE toward target = maximize target logit
            model.zero_grad(set_to_none=True)
            loss.backward()

            raw_grad = x_adv_alive.grad.detach()
            l1_norm = raw_grad.abs().sum(dim=1, keepdim=True).clamp(min=1e-12)
            grad_momentum[still_alive] = momentum * grad_momentum[still_alive] + raw_grad / l1_norm

            grad_sign = torch.sign(grad_momentum[still_alive])
            alpha_t = alpha * (0.3 + 0.7 * (1 + math.cos(math.pi * step / num_steps)) / 2)

            x_step = x_adv[still_alive].detach()
            x_step[:, cont_idx_t] = x_step[:, cont_idx_t] + alpha_t * grad_sign[:, cont_idx_t]
            # Project back into epsilon-ball around original x and into valid feature range
            x_step[:, cont_idx_t] = torch.max(
                torch.min(x_step[:, cont_idx_t],
                          torch.min(x[still_alive][:, cont_idx_t] + epsilon, x_max_t[cont_idx_t].unsqueeze(0))),
                torch.max(x[still_alive][:, cont_idx_t] - epsilon, x_min_t[cont_idx_t].unsqueeze(0)))
            x_adv[still_alive] = x_step

            with torch.no_grad():
                preds_alive = model(x_adv[still_alive]).argmax(dim=1)
                newly_fooled = still_alive.clone()
                newly_fooled[still_alive] = (preds_alive == target_class)
                still_alive[newly_fooled] = False

        with torch.no_grad():
            logits_final = model(x_adv)
            preds_final  = logits_final.argmax(dim=1)
            fooled_now   = (preds_final == target_class)
            per_loss = -F.cross_entropy(logits_final,
                torch.full((N,), target_class, dtype=torch.long, device=device), reduction="none")

        update_mask = (
            (fooled_now & ~best_is_fooled)
            | (fooled_now & best_is_fooled & (per_loss > best_loss))
            | (~fooled_now & ~best_is_fooled & (per_loss > best_loss))
        )
        best_is_fooled = torch.where(update_mask, fooled_now, best_is_fooled)
        best_loss      = torch.where(update_mask, per_loss, best_loss)
        best_x_adv     = torch.where(update_mask.unsqueeze(1).expand_as(x_adv), x_adv, best_x_adv)

    return best_x_adv.detach(), best_is_fooled

In [ ]:
EPSILONS = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 20]
TARGET_CLASSES = {1: "DoS_ATTACK", 2: "NON_DoS_ATTACK"}

results = []

for target_class, target_name in TARGET_CLASSES.items():
    print(f"\n{'='*60}")
    print(f"Target: {target_name} (class {target_class})")
    print(f"{'='*60}")
    for epsilon in EPSILONS:
        x_adv, fooled = targeted_pgd(
            model, X_benign_t, y_benign_t,
            target_class=target_class,
            epsilon=epsilon,
            x_min_t=x_min_t, x_max_t=x_max_t,
        )
        asr = fooled.float().mean().item()
        n_fooled = fooled.sum().item()

        with torch.no_grad():
            final_preds = model(x_adv).argmax(dim=1).cpu().numpy()

        results.append({
            "target_class": target_name,
            "epsilon": epsilon,
            "ASR": round(asr, 4),
            "n_fooled": n_fooled,
            "n_total": len(X_benign_t),
        })
        print(f"  epsilon={epsilon:>5}  ASR={asr*100:5.1f}%  ({n_fooled:,}/{len(X_benign_t):,} samples fooled)")

results_df = pd.DataFrame(results)
print("\n\nSummary:")
print(results_df.to_string(index=False))


Target: DoS_ATTACK (class 1)


In [ ]:
# ASR vs epsilon curves for both target classes
fig, ax = plt.subplots(figsize=(9, 5))

for target_name in TARGET_CLASSES.values():
    sub = results_df[results_df["target_class"] == target_name]
    ax.plot(sub["epsilon"], sub["ASR"] * 100, marker="o", label=f"→ {target_name}")

ax.set_xlabel("Epsilon (perturbation budget)")
ax.set_ylabel("Attack Success Rate (%)")
ax.set_title("Targeted Attack on BENIGN: ASR vs Epsilon")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig("benign_targeted_asr.png", dpi=150)
plt.show()

In [ ]:
print("Classification report on BENIGN samples after targeted attack (highest epsilon)")
all_labels = list(range(len(le.classes_)))
for target_class, target_name in TARGET_CLASSES.items():
    eps = EPSILONS[-1]
    x_adv, _ = targeted_pgd(
        model, X_benign_t, y_benign_t,
        target_class=target_class,
        epsilon=eps,
        x_min_t=x_min_t, x_max_t=x_max_t,
    )
    with torch.no_grad():
        preds = model(x_adv).argmax(dim=1).cpu().numpy()
    y_true = y_benign_t.cpu().numpy()
    print(f"\n--- Target: {target_name}  |  epsilon={eps} ---")
    print(classification_report(y_true, preds, target_names=le.classes_,
                                labels=all_labels, zero_division=0))

In [ ]:
# Export perturbed benign samples at each epsilon for further analysis
for target_class, target_name in TARGET_CLASSES.items():
    for epsilon in EPSILONS:
        x_adv, fooled = targeted_pgd(
            model, X_benign_t, y_benign_t,
            target_class=target_class,
            epsilon=epsilon,
            x_min_t=x_min_t, x_max_t=x_max_t,
        )
        with torch.no_grad():
            preds = model(x_adv).argmax(dim=1).cpu().numpy()

        x_adv_np = x_adv.cpu().numpy()
        adv_df = pd.DataFrame(scaler.inverse_transform(x_adv_np), columns=X.columns).round(0).astype(int)
        adv_df["true_label"] = 0        # BENIGN
        adv_df["target_class"] = target_class
        adv_df["y_pred"] = preds
        adv_df["fooled"] = fooled.cpu().numpy().astype(int)

        tag = target_name.lower().replace("_", "")
        filename = f"benign_targeted_{tag}_epsilon_{epsilon}.csv"
        adv_df.to_csv(filename, index=False)
        print(f"Saved {filename}  ({fooled.sum().item():,} fooled / {len(adv_df):,} total)")